# ⚡ 日本の電力消費量で学ぶ時系列分析
## AR → MA → ARIMA → SARIMA → ARIMAX

**データ出典：** Our World in Data "Energy Data" (CC BY 4.0) × 気象庁月平均気温  
年次データを実際の月別季節パターン（電力調査統計ベース）で展開した月次パネル

**前回授業との接続：** 前回は「気温 → 電力消費」を OLS 回帰で分析した。  
しかし電力データには **自己相関**（先月が多ければ今月も多い）と **季節性** がある。  
OLS はこれらを完全に無視している。今回はそれを直す。

| 章 | モデル | 所要時間 |
|----|--------|---------|
| 1 | データ確認・定常性・ACF/PACF | 20分 |
| 2 | AR(p) | 15分 |
| 3 | MA(q) | 10分 |
| 4 | ARIMA(p,d,q) | 15分 |
| 5 | SARIMA(p,d,q)(P,D,Q)s | 15分 |
| 6 | ARIMAX（外生変数：気温・構造変化ダミー） | 15分 |

---

## 0. ライブラリのインポート

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# statsmodels ── 時系列分析の本命
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose

from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

import matplotlib
for font in ['IPAexGothic', 'Noto Sans CJK JP', 'Hiragino Sans']:
    try: matplotlib.rc('font', family=font); break
    except: pass
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

print('準備完了 ✅')
print(f'statsmodels {sm.__version__}')


---
## 1. データ取得・可視化・定常性の確認

### 1-1. データ構築

Our World in Data の年次電力需要データを、  
実際の電力調査統計から得た月別季節パターンで月次に展開します。  
また東日本大震災（2011-2013）とコロナ禍（2020-2021）による需要変化を組み込みます。


In [ ]:
# ── 年次データ取得（Our World in Data）─────────────────────
OWID_URL = 'https://raw.githubusercontent.com/owid/energy-data/master/owid-energy-data.csv'
owid = pd.read_csv(OWID_URL)
jp_annual = (owid[owid['country'] == 'Japan']
             [['year', 'electricity_demand', 'gdp']]
             .dropna(subset=['electricity_demand'])
             .query('2005 <= year <= 2023')
             .copy())

# ── 月別季節シェア（電力調査統計ベース）─────────────────────
# 夏（7-8月）・冬（1月）にピーク、春秋に谷
monthly_share_raw = np.array([
    0.0893, 0.0814, 0.0804, 0.0755, 0.0765, 0.0797,
    0.0913, 0.0963, 0.0854, 0.0785, 0.0805, 0.0853
])
monthly_share = monthly_share_raw / monthly_share_raw.sum()

# 東京の月平均気温（℃、平年値）
avg_temp_clim = np.array([5.2, 5.7, 8.7, 14.6, 19.0, 22.4,
                           26.4, 27.1, 23.5, 17.5, 12.5, 7.8])

# ── 月次データに展開 ────────────────────────────────────────
rows = []
for _, row in jp_annual.iterrows():
    yr = int(row['year'])
    annual_twh = row['electricity_demand']
    # 構造変化：震災節電・コロナ
    struct_factor = (0.95 if yr in [2011, 2012]
                     else 0.97 if yr in [2013, 2020, 2021]
                     else 1.0)
    np.random.seed(yr * 13)
    noise = np.random.normal(0, 0.008, 12)
    temp_noise = np.random.normal(0, 0.5, 12)
    for m in range(1, 13):
        share = (monthly_share[m-1] + noise[m-1]) * struct_factor
        rows.append({
            'date':             pd.Timestamp(yr, m, 1),
            'electricity_twh':  annual_twh * share,
            'avg_temp':         avg_temp_clim[m-1] + temp_noise[m-1],
            'year': yr, 'month': m,
            'quake': 1 if yr in [2011, 2012, 2013] else 0,
            'covid': 1 if yr in [2020, 2021]        else 0,
        })

df = pd.DataFrame(rows).set_index('date').sort_index()
df.index.freq = 'MS'

print(f'月次データ: {len(df):,} ヶ月  ({df.index[0].strftime("%Y/%m")} 〜 {df.index[-1].strftime("%Y/%m")})')
df[['electricity_twh', 'avg_temp']].describe().round(2)


### 1-2. 時系列プロット

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

# 電力消費量
ax = axes[0]
ax.plot(df.index, df['electricity_twh'], color='#2980b9', lw=1.2, alpha=0.9)
ax.fill_between(df.index, df['electricity_twh'],
                df['electricity_twh'].min() * 0.95, alpha=0.12, color='#2980b9')
# 構造変化のハイライト
for yr_range, color, label in [
    (('2011-01','2013-12'), '#e74c3c', '震災節電'),
    (('2020-01','2021-12'), '#e67e22', 'コロナ禍'),
]:
    ax.axvspan(pd.Timestamp(yr_range[0]), pd.Timestamp(yr_range[1]),
               alpha=0.12, color=color, label=label)
ax.set(ylabel='電力消費量（TWh）', title='日本の月別電力消費量（2005-2023）')
ax.legend(fontsize=9)

# 月平均気温
ax = axes[1]
ax.plot(df.index, df['avg_temp'], color='#e74c3c', lw=1.0, alpha=0.8)
ax.axhline(df['avg_temp'].mean(), color='gray', lw=0.8, linestyle='--',
           label=f'平均 {df["avg_temp"].mean():.1f}°C')
ax.set(ylabel='月平均気温（℃）', xlabel='', title='東京の月平均気温')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print('💡 観察ポイント:')
print('  - 夏・冬のダブルピーク（エアコン需要）')
print('  - 2011-2012年: 震災後の節電で消費量が急低下')
print('  - 長期的には緩やかな低下トレンド（省エネ化）')


### 1-3. 季節分解（STL/古典的加法モデル）

時系列を **トレンド・季節性・残差** に分解して、構造を把握します。


In [ ]:
decomp = seasonal_decompose(df['electricity_twh'], model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(13, 9), sharex=True)
components = [
    (df['electricity_twh'], '原系列', '#2980b9'),
    (decomp.trend,          'トレンド', '#27ae60'),
    (decomp.seasonal,       '季節成分', '#e67e22'),
    (decomp.resid,          '残差', '#95a5a6'),
]
for ax, (data, title, color) in zip(axes, components):
    ax.plot(data, color=color, lw=1.2)
    ax.set_ylabel(title, fontsize=10)
    ax.axhline(0, color='black', lw=0.5, linestyle='--', alpha=0.4)

axes[0].set_title('季節分解（加法モデル）', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# 季節成分の月別平均
seasonal_monthly = decomp.seasonal.groupby(decomp.seasonal.index.month).mean()
print('月別季節成分（平均）:')
month_names = ['1月','2月','3月','4月','5月','6月',
               '7月','8月','9月','10月','11月','12月']
for m, v in zip(month_names, seasonal_monthly.values):
    bar = '█' * int(abs(v)/0.3) if abs(v) > 0 else ''
    sign = '+' if v >= 0 else ''
    print(f'  {m}: {sign}{v:5.2f} TWh  {bar}')


### 1-4. 定常性の検定

ARIMA 系モデルは**定常性**を前提とします。  
まず ADF 検定（単位根あり → 非定常）と KPSS 検定（定常 → 非定常）で確認します。

$$\text{ADF}:\; H_0: \text{単位根あり（非定常）} \qquad \text{KPSS}:\; H_0: \text{定常}$$


In [ ]:
def stationarity_report(series, name='series'):
    adf = adfuller(series.dropna(), autolag='AIC')
    kpss_stat, kpss_p, _, _ = kpss(series.dropna(), regression='ct', nlags='auto')
    print(f'--- {name} ---')
    print(f'  ADF  : t={adf[0]:7.3f}  p={adf[1]:.4f}  '
          f'→ {"定常 ✅" if adf[1] < 0.05 else "非定常 ⚠️"}')
    print(f'  KPSS : t={kpss_stat:7.3f}  p={kpss_p:.4f}  '
          f'→ {"非定常 ⚠️" if kpss_p < 0.05 else "定常 ✅"}')
    return adf[1], kpss_p

y = df['electricity_twh']
print('【原系列】')
p_adf, p_kpss = stationarity_report(y, '電力消費量（原系列）')

print()
print('【1階差分】')
p_adf_d1, _ = stationarity_report(y.diff().dropna(), '電力消費量（1階差分）')

print()
print('【季節差分（12ヶ月）】')
p_adf_ds, _ = stationarity_report(y.diff(12).dropna(), '電力消費量（季節差分）')

print()
print('【1階＋季節差分】')
p_adf_d1s, _ = stationarity_report(y.diff().diff(12).dropna(), '電力消費量（1階＋季節差分）')


### 1-5. ACF / PACF プロット（モデル次数の手がかり）

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

plot_acf( y,          lags=36, ax=axes[0,0], title='ACF（原系列）',       alpha=0.05)
plot_pacf(y,          lags=36, ax=axes[0,1], title='PACF（原系列）',      alpha=0.05)
plot_acf( y.diff(12).dropna(),  lags=36, ax=axes[1,0],
         title='ACF（季節差分後）',  alpha=0.05)
plot_pacf(y.diff(12).dropna(),  lags=36, ax=axes[1,1],
         title='PACF（季節差分後）', alpha=0.05)

for ax in axes.flatten():
    ax.set_xlabel('ラグ（月）')
plt.suptitle('ACF / PACF プロット', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('【ACF/PACF の読み方】')
print('  AR(p)  → ACF が指数減衰、PACF が p ラグ後に打ち切り')
print('  MA(q)  → ACF が q ラグ後に打ち切り、PACF が指数減衰')
print('  ARMA   → ACF・PACF ともに指数減衰')
print('  lag=12 に強いスパイク → 季節成分（S=12）')


---
## 2. AR(p) モデル ― 自己回帰

$$y_t = c + \phi_1 y_{t-1} + \phi_2 y_{t-2} + \cdots + \phi_p y_{t-p} + \varepsilon_t$$

「今月の電力消費量は、**過去 p ヶ月の電力消費量**の線形結合で説明できる」というモデル。

**直感：** 先月が寒くてたくさん使ったなら、今月も使う傾向がある。


In [ ]:
# 訓練・テスト分割（最後の24ヶ月をテスト）
train = df['electricity_twh'][:'2021-12']
test  = df['electricity_twh']['2022-01':]

print(f'訓練: {len(train)} ヶ月  テスト: {len(test)} ヶ月')

# p = 1, 2, 3, 12 で AIC 比較
print('\nAR(p) の AIC 比較（ARIMA(p,0,0) として推定）:')
results_ar = {}
for p in [1, 2, 3, 6, 12]:
    m = ARIMA(train, order=(p, 0, 0)).fit()
    results_ar[p] = m
    print(f'  AR({p:2d}): AIC={m.aic:8.1f}  BIC={m.bic:8.1f}')


In [ ]:
# AR(1) vs AR(12) の係数確認
print('AR(1) の推定結果:')
print(results_ar[1].summary().tables[1])
print()
print('AR(12) の推定結果:')
print(results_ar[12].summary().tables[1])


In [ ]:
# AR(1) の予測（テスト期間）
best_p = min(results_ar, key=lambda p: results_ar[p].aic)
m_ar_best = ARIMA(train, order=(best_p, 0, 0)).fit()
forecast_ar = m_ar_best.forecast(steps=len(test))

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(train.index[-36:], train[-36:],  color='#2980b9', lw=1.2, label='訓練データ')
ax.plot(test.index,        test,          color='#27ae60', lw=1.5, label='実測値')
ax.plot(test.index,        forecast_ar,  color='#e74c3c', lw=1.5, linestyle='--',
        label=f'AR({best_p}) 予測')
ax.set(title=f'AR({best_p}) モデルの予測（テスト期間: 2022-2023）',
       ylabel='電力消費量（TWh）')
ax.legend()
plt.tight_layout()
plt.show()

mae_ar = mean_absolute_error(test, forecast_ar)
rmse_ar = np.sqrt(mean_squared_error(test, forecast_ar))
print(f'AR({best_p}) MAE={mae_ar:.2f} TWh  RMSE={rmse_ar:.2f} TWh')
print(f'（平均値: {test.mean():.1f} TWh → 誤差率 {mae_ar/test.mean()*100:.1f}%）')
print()
print('💡 AR は季節性を捉えられていないため、夏冬の予測精度が低い')


---
## 3. MA(q) モデル ― 移動平均

$$y_t = \mu + \varepsilon_t + \theta_1 \varepsilon_{t-1} + \cdots + \theta_q \varepsilon_{t-q}$$

「今月の電力消費量は、**過去 q ヶ月の予測誤差（ショック）**の影響を受ける」というモデル。

**AR との違い：**  
- AR → 過去の**値**が影響  
- MA → 過去の**ショック**（誤差）が影響

**ACF の打ち切りで q を決める。**


In [ ]:
print('MA(q) の AIC 比較:')
results_ma = {}
for q in [1, 2, 3, 6, 12]:
    m = ARIMA(train, order=(0, 0, q)).fit()
    results_ma[q] = m
    print(f'  MA({q:2d}): AIC={m.aic:8.1f}  BIC={m.bic:8.1f}')


In [ ]:
# AR と MA の ACF/PACF の違いを可視化
fig, axes = plt.subplots(2, 2, figsize=(13, 7))

# AR(1) の理論 ACF/PACF
phi = results_ar[1].params.iloc[1]
lags = np.arange(0, 25)
acf_ar_theory  = phi ** lags
pacf_ar_theory = np.where(lags == 1, phi, np.where(lags == 0, 1, 0))

axes[0,0].bar(lags, acf_ar_theory, color='#2980b9', alpha=0.7)
axes[0,0].set(title=f'AR(1) 理論 ACF（φ={phi:.3f}）', xlabel='ラグ', ylabel='ACF')
axes[0,0].axhline(0, color='black', lw=0.5)

axes[0,1].bar(lags, pacf_ar_theory, color='#2980b9', alpha=0.7)
axes[0,1].set(title='AR(1) 理論 PACF（1ラグで打ち切り）', xlabel='ラグ', ylabel='PACF')
axes[0,1].axhline(0, color='black', lw=0.5)

# MA(1) の理論 ACF/PACF
theta = results_ma[1].params.iloc[1] if len(results_ma[1].params) > 1 else -0.4
acf_ma_theory  = np.where(lags == 0, 1, np.where(lags == 1, theta/(1+theta**2), 0))
pacf_ma_theory = [(-theta)**l * theta/(1+theta**2) if l > 0 else 1 for l in lags]

axes[1,0].bar(lags, acf_ma_theory, color='#e74c3c', alpha=0.7)
axes[1,0].set(title='MA(1) 理論 ACF（1ラグで打ち切り）', xlabel='ラグ', ylabel='ACF')
axes[1,0].axhline(0, color='black', lw=0.5)

axes[1,1].bar(lags[:15], pacf_ma_theory[:15], color='#e74c3c', alpha=0.7)
axes[1,1].set(title='MA(1) 理論 PACF（指数的減衰）', xlabel='ラグ', ylabel='PACF')
axes[1,1].axhline(0, color='black', lw=0.5)

plt.suptitle('AR(1) vs MA(1)：ACF/PACF の理論的パターン比較', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('AR と MA の識別ルール:')
print('  ACF が打ち切り  → MA モデル（q = 打ち切り点）')
print('  PACF が打ち切り → AR モデル（p = 打ち切り点）')
print('  両方が減衰      → ARMA モデル')


---
## 4. ARIMA(p,d,q) ― 差分を組み込む

$$\Delta^d y_t = c + \sum_{i=1}^p \phi_i \Delta^d y_{t-i} + \varepsilon_t + \sum_{j=1}^q \theta_j \varepsilon_{t-j}$$

**d**：非定常性への対処として**差分の階数**を指定。  
- $d=0$：定常（差分不要）  
- $d=1$：1階差分（ランダムウォーク的な非定常を除去）

先ほどの ADF 検定で d=1 が必要と判明した。  
残る次数 p, q は **AIC/BIC** で選ぶ。


In [ ]:
# ── 次数選択：グリッドサーチ ─────────────────────────────────
print('ARIMA(p,1,q) グリッドサーチ（AIC 最小を探す）:')
print(f'  {"(p,d,q)":12s}  {"AIC":>9}  {"BIC":>9}')
print('  ' + '-' * 35)

best_aic = np.inf
best_order = None
grid_results = []

for p in range(0, 4):
    for q in range(0, 4):
        try:
            m = ARIMA(train, order=(p, 1, q)).fit()
            grid_results.append({'order': (p,1,q), 'aic': m.aic, 'bic': m.bic})
            marker = ' ← best AIC' if m.aic < best_aic else ''
            if m.aic < best_aic:
                best_aic = m.aic
                best_order = (p, 1, q)
            print(f'  {str((p,1,q)):12s}  {m.aic:9.1f}  {m.bic:9.1f}{marker}')
        except:
            pass

print(f'\n最良モデル: ARIMA{best_order}  AIC={best_aic:.1f}')


In [ ]:
# ── 最良 ARIMA モデルの推定と診断 ──────────────────────────
m_arima = ARIMA(train, order=best_order).fit()
print(m_arima.summary())


In [ ]:
# ── ARIMA の診断プロット ────────────────────────────────────
fig = m_arima.plot_diagnostics(figsize=(13, 8))
plt.suptitle(f'ARIMA{best_order} 診断プロット', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ── ARIMA 予測 ─────────────────────────────────────────────
forecast_res = m_arima.get_forecast(steps=len(test))
pred_arima   = forecast_res.predicted_mean
ci_arima     = forecast_res.conf_int(alpha=0.05)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(train.index[-36:], train[-36:], color='#2980b9', lw=1.2, label='訓練データ')
ax.plot(test.index, test,              color='#27ae60', lw=1.5, label='実測値')
ax.plot(test.index, pred_arima,        color='#e74c3c', lw=1.5, linestyle='--',
        label=f'ARIMA{best_order} 予測')
ax.fill_between(test.index, ci_arima.iloc[:,0], ci_arima.iloc[:,1],
                color='#e74c3c', alpha=0.15, label='95% 予測区間')
ax.set(title=f'ARIMA{best_order} の予測', ylabel='電力消費量（TWh）')
ax.legend()
plt.tight_layout()
plt.show()

mae_arima = mean_absolute_error(test, pred_arima)
rmse_arima = np.sqrt(mean_squared_error(test, pred_arima))
print(f'ARIMA{best_order}  MAE={mae_arima:.2f} TWh  RMSE={rmse_arima:.2f} TWh')
print(f'AR({best_order[0]})  MAE={mae_ar:.2f} → ARIMA MAE={mae_arima:.2f}（改善: {mae_ar-mae_arima:.2f}）')
print()
print('⚠️  ARIMA はまだ季節性を捉えられていない → 夏冬の予測誤差が大きい')


---
## 5. SARIMA(p,d,q)(P,D,Q)s ― 季節性を明示的にモデル化

$$\Phi_P(B^s)\phi_p(B)\Delta^D_s \Delta^d y_t = \Theta_Q(B^s)\theta_q(B)\varepsilon_t$$

電力消費には **12ヶ月の季節周期**（夏・冬のピーク）が明確に存在する。  
ARIMA に季節成分 $(P, D, Q)_s$ を追加して対処する。

| 記号 | 意味 |
|------|------|
| $s=12$ | 季節周期（月次データ） |
| $D=1$ | 季節差分（$y_t - y_{t-12}$）で季節性を除去 |
| $P$ | 季節 AR 次数（lag=12 の自己回帰） |
| $Q$ | 季節 MA 次数 |


In [ ]:
# ── SARIMA 次数選択 ──────────────────────────────────────────
print('SARIMA(p,1,q)(P,1,Q)12 グリッドサーチ:')
print(f'  {"非季節(p,q)":12s}  {"季節(P,Q)":10s}  {"AIC":>9}')
print('  ' + '-' * 38)

best_sarima_aic = np.inf
best_sarima_order = None
sarima_results = []

for p in range(0, 3):
    for q in range(0, 3):
        for P in range(0, 3):
            for Q in range(0, 3):
                try:
                    m = SARIMAX(train,
                                order=(p, 1, q),
                                seasonal_order=(P, 1, Q, 12)).fit(disp=False)
                    sarima_results.append({
                        'order': (p,1,q), 'seasonal': (P,1,Q,12),
                        'aic': m.aic, 'bic': m.bic, 'model': m
                    })
                    if m.aic < best_sarima_aic:
                        best_sarima_aic = m.aic
                        best_sarima_order = {'order':(p,1,q), 'seasonal':(P,1,Q,12)}
                except:
                    pass

# AIC 上位5モデルを表示
sarima_df = pd.DataFrame(sarima_results).sort_values('aic')
for _, row in sarima_df.head(5).iterrows():
    print(f'  {str(row["order"]):12s}  {str(row["seasonal"][:3]):10s}  {row["aic"]:9.1f}')

print(f'\n最良: SARIMA{best_sarima_order["order"]}x{best_sarima_order["seasonal"]}  AIC={best_sarima_aic:.1f}')
print(f'ARIMA{best_order} AIC={best_aic:.1f} → 改善: {best_aic - best_sarima_aic:.1f}')


In [ ]:
# ── 最良 SARIMA の推定 ──────────────────────────────────────
best_row = sarima_df.iloc[0]
m_sarima = best_row['model']
print(m_sarima.summary())


In [ ]:
# ── SARIMA 予測と比較 ────────────────────────────────────────
fc_sarima = m_sarima.get_forecast(steps=len(test))
pred_sarima = fc_sarima.predicted_mean
ci_sarima   = fc_sarima.conf_int(alpha=0.05)

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# 全期間
ax = axes[0]
ax.plot(df.index, df['electricity_twh'], color='#2980b9', lw=1.0, alpha=0.6, label='実測値')
ax.plot(test.index, pred_sarima, color='#e74c3c', lw=1.5, linestyle='--',
        label=f'SARIMA 予測')
ax.fill_between(test.index, ci_sarima.iloc[:,0], ci_sarima.iloc[:,1],
                color='#e74c3c', alpha=0.15)
ax.set(ylabel='TWh', title='全期間')
ax.legend()

# テスト期間拡大
ax = axes[1]
ax.plot(test.index, test.values, color='#27ae60', lw=2, label='実測値', marker='o', ms=4)
ax.plot(test.index, pred_arima.values, color='#95a5a6', lw=1.5, linestyle='--',
        label=f'ARIMA{best_order}')
ax.plot(test.index, pred_sarima.values, color='#e74c3c', lw=2, linestyle='--',
        label=f'SARIMA（最良）')
ax.fill_between(test.index, ci_sarima.iloc[:,0], ci_sarima.iloc[:,1],
                color='#e74c3c', alpha=0.12)
ax.set(ylabel='TWh', title='テスト期間（2022-2023）拡大')
ax.legend()

plt.suptitle('SARIMA vs ARIMA：季節性の捕捉比較', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

mae_sarima = mean_absolute_error(test, pred_sarima)
rmse_sarima = np.sqrt(mean_squared_error(test, pred_sarima))
print('【モデル比較（テスト期間 MAE）】')
print(f'  ARIMA        MAE={mae_arima:.2f} TWh  ({mae_arima/test.mean()*100:.1f}%)')
print(f'  SARIMA (最良) MAE={mae_sarima:.2f} TWh  ({mae_sarima/test.mean()*100:.1f}%)')
print(f'  改善幅: {mae_arima - mae_sarima:.2f} TWh')


---
## 6. ARIMAX ― 外生変数を追加する

SARIMAX の `exog` 引数に外生変数を追加することで、  
純粋な時系列パターン以外の情報（気温・イベントダミー等）を取り込む。

$$\text{SARIMAX}(\ldots) + \beta_1 \cdot T_t + \beta_2 \cdot \text{quake}_t + \beta_3 \cdot \text{covid}_t$$

### ❓ 問題：前回授業との接続

前回、OLS 回帰で「気温 → 電力消費」を分析した。  
今回は気温を **外生変数**として SARIMA に組み込む。  
OLS の回帰係数と ARIMAX の気温係数は一致するか？なぜ違うか？


In [ ]:
# ── 外生変数の準備 ──────────────────────────────────────────
exog_cols = ['avg_temp', 'quake', 'covid']
exog_train = df.loc[train.index, exog_cols]
exog_test  = df.loc[test.index,  exog_cols]

best_o = best_sarima_order['order']
best_s = best_sarima_order['seasonal']

# ARIMAX（気温のみ）
m_arimax_temp = SARIMAX(train, exog=exog_train[['avg_temp']],
                         order=best_o, seasonal_order=best_s).fit(disp=False)
# ARIMAX（気温 + 構造変化ダミー）
m_arimax_full = SARIMAX(train, exog=exog_train,
                         order=best_o, seasonal_order=best_s).fit(disp=False)

print('【モデル比較（AIC）】')
print(f'  SARIMA（外生変数なし）  AIC={m_sarima.aic:.1f}')
print(f'  ARIMAX（気温のみ）      AIC={m_arimax_temp.aic:.1f}')
print(f'  ARIMAX（気温＋ダミー）  AIC={m_arimax_full.aic:.1f}')


In [ ]:
# ── ARIMAX 結果の詳細 ──────────────────────────────────────
print(m_arimax_full.summary())


In [ ]:
# ── 外生変数の係数を OLS と比較 ─────────────────────────────
import statsmodels.formula.api as smf

ols_model = smf.ols('electricity_twh ~ avg_temp + quake + covid',
                    data=df.loc[train.index]).fit()

print('【気温係数の比較】')
print(f'  OLS 回帰:  β(気温) = {ols_model.params["avg_temp"]:+.4f} TWh/°C  '
      f'(p={ols_model.pvalues["avg_temp"]:.4f})')
temp_coef = m_arimax_full.params.get('avg_temp', m_arimax_full.params.iloc[-3])
temp_pval = m_arimax_full.pvalues.get('avg_temp', m_arimax_full.pvalues.iloc[-3])
print(f'  ARIMAX:    β(気温) = {temp_coef:+.4f} TWh/°C  (p={temp_pval:.4f})')
print()
print('💡 なぜ係数が異なるか？')
print('  OLS は時系列の自己相関を無視するため、気温の効果が「余分」に割り振られる。')
print('  ARIMAX は時系列構造を制御した上で気温の純粋な効果を推定する。')


In [ ]:
# ── 全モデルの最終比較 ──────────────────────────────────────
fc_arimax = m_arimax_full.get_forecast(steps=len(test), exog=exog_test)
pred_arimax = fc_arimax.predicted_mean
ci_arimax   = fc_arimax.conf_int(alpha=0.05)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(test.index, test.values, color='#27ae60', lw=2.5, label='実測値', zorder=5)
for pred, label, color, ls in [
    (pred_arima,  f'ARIMA{best_order}',    '#95a5a6', '--'),
    (pred_sarima, 'SARIMA（最良）',          '#e67e22', '--'),
    (pred_arimax, 'ARIMAX（最良＋外生変数）', '#e74c3c', '-'),
]:
    ax.plot(test.index, pred.values, color=color, lw=1.8, linestyle=ls, label=label)
ax.fill_between(test.index, ci_arimax.iloc[:,0], ci_arimax.iloc[:,1],
                color='#e74c3c', alpha=0.12, label='ARIMAX 95% CI')
ax.set(title='全モデル予測比較（テスト期間 2022-2023）', ylabel='電力消費量（TWh）')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# 誤差比較表
mae_arimax  = mean_absolute_error(test, pred_arimax)
rmse_arimax = np.sqrt(mean_squared_error(test, pred_arimax))

comparison = pd.DataFrame({
    'モデル':  [f'AR({best_order[0]})', f'ARIMA{best_order}', 'SARIMA（最良）', 'ARIMAX（全外生変数）'],
    'MAE':     [mae_ar, mae_arima, mae_sarima, mae_arimax],
    'RMSE':    [rmse_ar, rmse_arima, rmse_sarima, rmse_arimax],
    '誤差率%': [mae_ar/test.mean()*100, mae_arima/test.mean()*100,
                mae_sarima/test.mean()*100, mae_arimax/test.mean()*100],
}).round(3)
print(comparison.to_string(index=False))


---
## 7. まとめと演習問題

### モデルの系譜と改善の流れ

```
AR(p) ─── MA(q) ─→ ARMA(p,q) ─→ ARIMA(p,d,q) ─→ SARIMA(p,d,q)(P,D,Q)s ─→ ARIMAX
                                        ↑差分追加        ↑季節成分追加          ↑外生変数追加
```

### 各モデルが捉えるもの

| モデル | 捉えるもの | 今回の電力データへの適用 |
|--------|-----------|------------------------|
| AR(p) | 短期自己相関 | 先月の電力消費が今月に影響 |
| MA(q) | 過去ショックの影響 | 節電キャンペーンの余韻 |
| ARIMA | 非定常トレンド + AR/MA | 長期的な省エネトレンド |
| SARIMA | 季節性 | 夏・冬のエアコン需要 |
| ARIMAX | 外生変数 | 気温・震災・コロナの効果 |

### 演習問題

1. **次数の感度分析**：SARIMA の $(p,q)$ を (0,0), (1,0), (0,1), (1,1) で変えたとき、  
   テスト期間の MAE はどう変わるか？最良次数は AIC の選択と一致するか？

2. **予測区間の意味**：95% 予測区間が12ヶ月後にどれだけ広がるか計算し、  
   「予測の不確実性は時間とともにどう変化するか」を説明せよ。

3. **外生変数の追加検討**：GDP（景気）を外生変数に追加したとき、  
   AIC と気温係数はどう変わるか？GDP と気温の間に多重共線性はあるか？

4. **構造変化モデル**：コロナダミー変数を除き、代わりに  
   `covid_trend = max(0, (date - 2020-01) / 12)` という「回復トレンド」を作り、  
   モデルに追加せよ。どちらが AIC で良いか？

5. **（発展）auto_arima の利用**：`pmdarima` ライブラリの `auto_arima` 関数を使い、  
   今回手動で選んだ次数と比較せよ。
   ```python
   from pmdarima import auto_arima
   result = auto_arima(train, seasonal=True, m=12, information_criterion='aic')
   ```
